# She Care AI: Internship Dropout Risk & Fairness Model

This notebook generates a synthetic dataset of 2,000 student records and trains a Random Forest Classifier to predict the risk of dropping out of an internship. It also demonstrates how to calculate the Gini Coefficient for fairness monitoring in the Policy Simulator.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
import joblib

## 1. Generate Synthetic Dataset
We simulate 2000 students with features that might influence their success in an internship. In a real-world scenario, this data would come from the user's uploaded resume and platform interactions.

In [ ]:
np.random.seed(42)
n_students = 2000

# Features based on our React app data
ai_match_score = np.random.normal(70, 15, n_students).clip(0, 100)
stipend_amount = np.random.uniform(0, 50000, n_students) # Monthly stipend in INR
living_cost_index = np.random.uniform(50, 150, n_students) # Regional cost of living index
commute_time_mins = np.random.uniform(10, 120, n_students)
mentorship_hours = np.random.normal(5, 2, n_students).clip(0, 20)
financial_need_score = np.random.uniform(1, 10, n_students) # 10 is highest need

# Calculate a latent "risk" score based on features
# High commute, low stipend in a high living cost area, and low AI match score increase risk.
risk_score = (
    (100 - ai_match_score) * 0.3 +
    (living_cost_index / stipend_amount.clip(1)) * 10000 * 0.2 +
    commute_time_mins * 0.1 +
    (10 - mentorship_hours) * 1.5 +
    financial_need_score * 2
)

# Convert to probabilities using sigmoid-like transformation
prob_dropout = 1 / (1 + np.exp(-(risk_score - np.median(risk_score)) / np.std(risk_score)))

# Binary outcome (1 = Dropped out, 0 = Completed)
dropped_out = (np.random.rand(n_students) < prob_dropout).astype(int)

df = pd.DataFrame({
    'ai_match_score': ai_match_score,
    'stipend_amount': stipend_amount,
    'living_cost_index': living_cost_index,
    'commute_time_mins': commute_time_mins,
    'mentorship_hours': mentorship_hours,
    'financial_need_score': financial_need_score,
    'dropped_out': dropped_out
})

print("Dataset Shape:", df.shape)
df.head()

## 2. Train Random Forest Classifier
We'll train a Random Forest model to predict the `dropped_out` target variable.

In [ ]:
X = df.drop('dropped_out', axis=1)
y = df['dropped_out']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

rf_model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
rf_model.fit(X_train, y_train)

y_pred = rf_model.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

## 3. Calculate Fairness Metric (Gini Coefficient)
To ensure our model doesn't unfairly target specific demographics (or to evaluate the distribution of risk scores), we can use the Gini Coefficient. 
A higher Gini indicates greater inequality in the distribution.

In [ ]:
def gini_coefficient(x):
    """Compute Gini coefficient of array of values"""
    diffsum = 0
    for i, xi in enumerate(x[:-1], 1):
        diffsum += np.sum(np.abs(xi - x[i:]))
    if len(x) == 0 or np.mean(x) == 0:
        return 0
    return diffsum / (len(x)**2 * np.mean(x))

# Let's look at the distribution of predicted probabilities
pred_probs = rf_model.predict_proba(X_test)[:, 1]
gini_val = gini_coefficient(pred_probs)
print(f"Gini Coefficient for predicted Dropout Risk Scores: {gini_val:.4f}")

## 4. Export the Model
Save the trained model so it can be loaded in the Vercel Serverless Function (`/api/predict-risk.py` or similar).

In [ ]:
joblib.dump(rf_model, 'dropout_risk_model.joblib')
print("Model saved to 'dropout_risk_model.joblib'")